# ⚖️ Legal GraphRAG - Sandbox (Ablation Study)

Questo notebook consente di testare e confrontare il comportamento dell'engine RAG in due configurazioni:
1. **RAG Semplice (Base)**: Solo Vector Search + BM25 Search. Senza l'ontologia TESEO e l'espansione delle citazioni.
2. **GraphRAG (Completo)**: Ibrido con Vector Search + BM25 + Ricerca semantica sui topic TESEO e navigazione ricorsiva delle citazioni (Multi-hop).

In [ ]:
import os, sys, asyncio, nest_asyncio
nest_asyncio.apply()
sys.path.append(os.getcwd())

import logging
# Disabilita i log prolissi
logging.basicConfig(level=logging.INFO)
logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)
logging.getLogger("httpx").setLevel(logging.WARNING)

from src.rag.engine import RagEngine

# Configurazione delle variabili d'ambiente
os.environ["NEO4J_URI"] = "bolt://localhost:7687"
os.environ["QWEN3_ENDPOINT"] = "http://localhost:11434"

engine = RagEngine()
print("✅ Engine RAG inizializzato correttamente.")

In [ ]:
import textwrap

def print_results(title, chunks):
    print(f"\n{'='*80}")
    print(f" {title.upper()} (Trovati {len(chunks)} chunk)")
    print(f"{'='*80}\n")
    
    if not chunks:
        print("  Nessun risultato trovato.\n")
        return
        
    for i, c in enumerate(chunks, 1):
        work_title = c.metadata.get("work_title", "Titolo non disponibile").strip()
        if len(work_title) > 80:
            work_title = work_title[:77] + "..."
        score = f"{c.score:.4f}"
        source = c.source.upper()
        context = c.structural_context or "N/A"
        
        header = f"[{i}] SCORE: {score} | FONTE: {source}"
        print(header)
        print(f"  ATTO: {work_title}")
        print(f"  POSIZIONE: {context}")
        print("  " + "-" * (len(header)))
        
        paragraphs = c.text.split('\n')
        for p in paragraphs:
            if p.strip():
                wrapped = textwrap.fill(p, width=100, initial_indent="    ", subsequent_indent="    ")
                print(wrapped)
        print(f"\n{'.' * 80}\n")

### ⚖️ Esecuzione Studio di Ablazione

Eseguiamo la stessa query nelle due modalità per confrontare i risultati.

In [ ]:
query = "Quali sono le competenze delle regioni?"

print(f"🔍 Esecuzione query di test: '{query}'\n")

# 1. Esecuzione RAG Semplice (Senza Graph e senza Multi-hop)
print("⏳ Esecuzione RAG Semplice...")
chunks_simple = await engine.retrieve(
    query=query,
    enable_graph_search=False,
    enable_multi_hop=False
)

# 2. Esecuzione GraphRAG (Completo)
print("⏳ Esecuzione GraphRAG Completo...")
chunks_graph = await engine.retrieve(
    query=query,
    enable_graph_search=True,
    enable_multi_hop=True
)

# Mostra i risultati a confronto
print_results("1. Risultati RAG Semplice (Solo Vettoriale + BM25)", chunks_simple)
print_results("2. Risultati GraphRAG (Con TESEO e Citazioni Multi-hop)", chunks_graph)

### 📊 Analisi delle Differenze

Identifichiamo quali chunk sono stati introdotti o migliorati grazie al Knowledge Graph.

In [ ]:
simple_ids = {c.expression_id for c in chunks_simple if c.expression_id}
graph_ids = {c.expression_id for c in chunks_graph if c.expression_id}

added_by_graph = [c for c in chunks_graph if c.expression_id not in simple_ids]

print(f"=== ANALISI ABLAZIONE ===")
print(f"Numero di chunk in RAG Semplice: {len(chunks_simple)}")
print(f"Numero di chunk in GraphRAG: {len(chunks_graph)}")
print(f"Chunk esclusivi del GraphRAG (grazie a TESEO o Citazioni): {len(added_by_graph)}\n")

if added_by_graph:
    for i, c in enumerate(added_by_graph, 1):
        source_detail = "Topic TESEO" if "graph" in c.source else "Espansione Citazioni (Multi-hop)"
        print(f"[{i}] ID: {c.expression_id} | Origine: {c.source.upper()} ({source_detail})")
        print(f"    Atto: {c.metadata.get('work_title', 'N/A')}")
        print(f"    Contesto: {c.structural_context}")
        snippet = c.text[:120].replace('\n', ' ').strip() + "..."
        print(f"    Snippet: {snippet}\n")
else:
    print("Nessun chunk esclusivo trovato (i due metodi hanno restituito gli stessi documenti).")

In [ ]:
await engine.close()
print("✅ Connessioni chiuse.")